In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.ticker import FuncFormatter
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.lines import Line2D

# ---------------------------------------------------
# Style FIRST (resets rcParams)
# ---------------------------------------------------
plt.style.use("default")

# ---------------------------------------------------
# Global font configuration (AFTER style)
# ---------------------------------------------------
font_size = 14
mpl.rcParams.update({
    "font.size": font_size,
    "axes.labelsize": font_size,
    "axes.titlesize": font_size,
    "xtick.labelsize": font_size,
    "ytick.labelsize": font_size,
    "legend.fontsize": font_size,
    "axes.linewidth": 1.3,
})

# ---------------------------------------------------
# Load dataset
# ---------------------------------------------------
df = pd.read_csv("datasets/MOU2D.csv", delimiter=";", engine="python")

# Use first THREE objectives
objs = df.values[:, -3:].astype(float)

# ---------------------------------------------------
# Objective directions
# True  -> minimize
# False -> maximize
# ---------------------------------------------------
minimize = np.array([False, True, False])

# ---------------------------------------------------
# Dominance check (mixed objectives)
# ---------------------------------------------------
def dominates(q, p, minimize):
    strictly_better = False

    for k in range(len(p)):
        if minimize[k]:
            if q[k] > p[k]:
                return False
            if q[k] < p[k]:
                strictly_better = True
        else:
            if q[k] < p[k]:
                return False
            if q[k] > p[k]:
                strictly_better = True

    return strictly_better

# ---------------------------------------------------
# Update Pareto front with a batch
# ---------------------------------------------------
def update_pareto_front(current_pf, batch, minimize):
    current_pf = list(current_pf)

    for b in batch:
        dominated = False
        to_remove = []

        for i, p in enumerate(current_pf):
            if dominates(p, b, minimize):
                dominated = True
                break
            if dominates(b, p, minimize):
                to_remove.append(i)

        if not dominated:
            for idx in reversed(to_remove):
                current_pf.pop(idx)
            current_pf.append(b)

    return current_pf

# ---------------------------------------------------
# Batch-wise Pareto computation
# ---------------------------------------------------
batch_size = 2000
pareto_frontier = []

for start in range(0, len(objs), batch_size):
    batch = objs[start:start + batch_size]
    pareto_frontier = update_pareto_front(pareto_frontier, batch, minimize)

pf = np.array(pareto_frontier)

print(f"Pareto front size: {len(pf)}")

# ---------------------------------------------------
# Plot (3D)
# ---------------------------------------------------
fig = plt.figure(figsize=(8, 6), constrained_layout=True)
ax = fig.add_subplot(111, projection="3d")

# All solutions
ax.scatter(
    objs[:, 0], objs[:, 1], objs[:, 2],
    color="grey",
    alpha=0.15,
    s=40,
    rasterized=True,
    edgecolors="black",
    depthshade=False,
    zorder=1
)

# Pareto front (ON TOP)
ax.scatter(
    pf[:, 0], pf[:, 1], pf[:, 2],
    color="red",
    s=150,
    marker="s",
    edgecolors="black",
    depthshade=False,
    zorder=100
)

# Axis labels
ax.set_xlabel("COP ($\\times 10^7 \\; \\mathrm{m}^3$)", labelpad=16)
ax.set_ylabel("CWP ($\\times 10^7 \\; \\mathrm{m}^3$)", labelpad=16)
ax.set_zlabel("NPV ($\\times 10^9 \\; \\mathrm{USD}$)", labelpad=16)

# Axis formatting
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e7:.1f}"))
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e7:.1f}"))
ax.zaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1e9:.1f}"))

ax.grid(False)
ax.tick_params(colors="black", width=1.3)

# Legend
legend_elements = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markerfacecolor="grey",
        markeredgecolor="black",
        markeredgewidth=0.6,
        markersize=8,
        label="All solutions"
    ),
    Line2D(
        [0], [0],
        marker="s",
        linestyle="",
        markerfacecolor="red",
        markeredgecolor="black",
        markeredgewidth=0.8,
        markersize=9,
        label="Non-dominated front"
    )
]

ax.legend(
    handles=legend_elements,
    loc="upper left",
    frameon=True
)

# View angle
ax.view_init(elev=30, azim=225)

plt.savefig("figures/MOU2D.pdf", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Extract objective columns
cop = [row[0] for row in objs]
cwp = [row[1] for row in objs]
npv = [row[2] for row in objs]

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=False)

# ---------- COP & CWP ----------
box1 = axes[0].boxplot(
    [cop, cwp],
    tick_labels=["COP", "CWP"],
    showfliers=True,
    patch_artist=True
)

colors = ["tab:blue", "tab:orange"]

for element in ["boxes", "whiskers", "caps", "medians"]:
    plt.setp(box1[element], color="black")

for patch, color in zip(box1["boxes"], colors):
    patch.set_facecolor(color)

axes[0].set_ylabel("Objective value ($\\times 10^7 \\mathrm{m}^3$)")
axes[0].yaxis.set_major_formatter(
    FuncFormatter(lambda y, _: f"{y/1e7:.1f}")
)
axes[0].grid(True, axis="y", alpha=0.4)

# ---------- NPV ----------
box2 = axes[1].boxplot(
    [npv],
    tick_labels=["NPV"],
    showfliers=True,
    patch_artist=True
)

for element in ["boxes", "whiskers", "caps", "medians"]:
    plt.setp(box2[element], color="black")

box2["boxes"][0].set_facecolor("tab:green")

axes[1].set_ylabel("Objective value ($\\times 10^9 \\; \\mathrm{USD}$)")
axes[1].yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f"{x/1e9:.1f}")
)
axes[1].grid(True, axis="y", alpha=0.4)

plt.tight_layout()
plt.savefig("figures/boxplot_MOU2D.pdf")
plt.show()


In [ ]:
df.head(10)

In [ ]:
display(df.dtypes)

In [ ]:
# Configura??es para exibi??o completa de DataFrames no pandas

# # Mostrar mais linhas
# pd.set_option('display.max_rows', 150)

# # Mostrar mais colunas
# pd.set_option('display.max_columns', None)

# # Mostrar a largura completa das colunas
# pd.set_option('display.max_colwidth', None)

# Resetar para padr?o (se necess?rio)
# pd.reset_option('display.max_rows')

## Preprocessing

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
   

preprocessor = Pipeline(steps=[
    ("scaler", StandardScaler())
])

## Regression

In [ ]:
#########################
# DATA
#########################

# dado
X = df.drop(columns=["COP",	"CWP", "NPV"]) 
y = df[["COP",	"CWP", "NPV"]]

In [ ]:
#########################
# MODELS
#########################

from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import os


algorithms = {

    "RF": (
        RandomForestRegressor(random_state=0),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [5, 10, None]
        }
    ),

    "GB": (
        MultiOutputRegressor(GradientBoostingRegressor(random_state=0)),
        {
            "model__estimator__n_estimators": [100, 200],
            "model__estimator__learning_rate": [0.05, 0.1]
        }
    ),

    "SVR": (
        MultiOutputRegressor(SVR()),
        {
            "model__estimator__C": [0.1, 1, 10, 50],
            "model__estimator__kernel": ["linear", "rbf"],
             "model__estimator__gamma": ["scale", "auto"]
        }
    ),

    "Ridge": (
        MultiOutputRegressor(Ridge()),
        {"model__estimator__alpha": [0.1, 1, 10]}
    ),

    "Lasso": (
        MultiOutputRegressor(Lasso(max_iter=5000)),
        {"model__estimator__alpha": [0.01, 0.1, 1]}
    ),

    "ElasticNet": (
        MultiOutputRegressor(ElasticNet(max_iter=5000)),
        {
            "model__estimator__alpha": [0.01, 0.1, 1],
            "model__estimator__l1_ratio": [0.1, 0.5, 0.9]
        }
    ),

    "KNN": (
        MultiOutputRegressor(KNeighborsRegressor()),
        {"model__estimator__n_neighbors": [3, 5, 7]}
    ),

    "CatBoost": (
    MultiOutputRegressor(
        CatBoostRegressor(
            loss_function="RMSE",
            random_seed=42,
            verbose=False,      # <-- SILENCIA LOGS
            thread_count=1      # <-- controle de paralelismo (fair comparison)
        )
    ),
    {
        "model__estimator__iterations": [300, 500],
        "model__estimator__learning_rate": [0.01, 0.05, 0.1],
        "model__estimator__depth": [6, 8, 10],
        "model__estimator__l2_leaf_reg": [1, 3, 5]
    }
),

    "XGBoost": (
    MultiOutputRegressor(
        XGBRegressor(
            objective="reg:squarederror",
            tree_method="hist",     
            device='cuda',       # for?a uso da GPU
            random_state=42,
            n_jobs=1
        )
    ),
    {
        "model__estimator__n_estimators": [200, 400],
        "model__estimator__max_depth": [3, 5, 7],
        "model__estimator__learning_rate": [0.01, 0.05, 0.1],
        "model__estimator__subsample": [0.8, 1.0],
        "model__estimator__colsample_bytree": [0.8, 1.0]
    }
    ),

    "LightGBM": (
        MultiOutputRegressor(
            LGBMRegressor(
                objective="regression",
                random_state=42,
                n_jobs=1,
                verbosity=-1, # silencia warnings
                device='gpu',       # for?a uso da GPU
                gpu_platform_id=0,  
                gpu_device_id=0     
            )
        ),
        {
            "model__estimator__n_estimators": [200, 400],
            "model__estimator__learning_rate": [0.01, 0.05, 0.1],
            "model__estimator__max_depth": [-1, 5, 10],
            "model__estimator__num_leaves": [31, 63, 127],
            "model__estimator__subsample": [0.8, 1.0],
            "model__estimator__colsample_bytree": [0.8, 1.0]
        }
    )
}


In [ ]:
###########################
# PIPELINE
###########################

import time
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings(
    "ignore",
    message=".*Falling back to prediction using DMatrix.*"
)

def run_multioutput_regression_cv(
    algorithms,
    X,
    y,
    preprocessor,
    n_splits=5
):
    scores = {}
    best_params = {}
    times = {}

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for name, (model, param_grid) in algorithms.items():

        print(f"Treinando {name}...")

        start_alg = time.time()

        fold_scores = []

        pipe = Pipeline(steps=[
            ("preprocess", preprocessor),
            ("model", model)
        ])

        grid = RandomizedSearchCV(
            estimator=pipe,
            param_distributions=param_grid,
            n_iter=10,
            scoring="neg_mean_squared_error",
            cv=cv,
            n_jobs=-1,
            verbose=2,
            random_state=42
        )

        start_search = time.time()
        grid.fit(X, y)
        end_search = time.time()
        search_time = end_search - start_search
        best_params[name] = grid.best_params_

        start_cv = time.time()
        # Avalia??o por fold
        for train_idx, test_idx in cv.split(X):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            grid.best_estimator_.fit(X_train, y_train)
            y_pred = grid.best_estimator_.predict(X_test)

            fold_scores.append({
                "mse": mean_squared_error(y_test, y_pred, multioutput="uniform_average"),
                "mae": mean_absolute_error(y_test, y_pred, multioutput="uniform_average"),
                "r2": r2_score(y_test, y_pred, multioutput="uniform_average")
            })
        end_cv = time.time()
        cv_time = end_cv - start_cv

        scores[name] = fold_scores

        # ---- SALVA RESULTADOS PARCIAIS ----
        mean_results = pd.DataFrame({
            alg: [
                np.mean([s['mse'] for s in scores[alg]]),
                np.mean([s['mae'] for s in scores[alg]]),
                np.mean([s['r2'] for s in scores[alg]])
            ]
            for alg in scores
        }, index=['MSE', 'MAE', 'R?'])

        std_results = pd.DataFrame({
            alg: [
                np.std([s['mse'] for s in scores[alg]]),
                np.std([s['mae'] for s in scores[alg]]),
                np.std([s['r2'] for s in scores[alg]])
            ]
            for alg in scores
        }, index=['MSE', 'MAE', 'R?'])

        mean_results.to_csv("/home/flopes/artigo_matheus_jan26/2026_bracis_datasets/results/results_MOU2D/regression_results_mean.csv")
        std_results.to_csv("/home/flopes/artigo_matheus_jan26/2026_bracis_datasets/results/results_MOU2D/regression_results_std.csv")
        pd.DataFrame(best_params).T.to_csv("/home/flopes/artigo_matheus_jan26/2026_bracis_datasets/results/results_MOU2D/regression_best_params.csv")

        end_alg = time.time()
        total_time = end_alg - start_alg

        times[name] = {
            "search_time_min": search_time / 60,
            "cv_time_min": cv_time / 60,
            "total_time_min": total_time / 60
        }

    return scores, best_params, times

In [ ]:
ini = time.time()
# Rodando a valida??o cross-validation
scores, best_params, times = run_multioutput_regression_cv(
    algorithms=algorithms,
    X=X,          
    y=y,         
    preprocessor=preprocessor,
    n_splits=5 
)

fim = time.time()
print('\nTempo TOTAL de execu??o:', round((fim - ini) / 60, 2), 'minutos')

times_df = pd.DataFrame(times).T.round(2)
times_df.to_csv("/home/flopes/artigo_matheus_jan26/2026_bracis_datasets/results/results_MOU2D/regression_times.csv")

print("\nTempo de execu??o por algoritmo (minutos)")
display(times_df)

mean_results = pd.DataFrame({
    alg: [
        np.mean([s['mse'] for s in scores[alg]]),
        np.mean([s['mae'] for s in scores[alg]]),
        np.mean([s['r2'] for s in scores[alg]])
    ]
    for alg in scores
}, index=['MSE', 'MAE', 'R?'])


std_results = pd.DataFrame({
    alg: [
        np.std([s['mse'] for s in scores[alg]]),
        np.std([s['mae'] for s in scores[alg]]),
        np.std([s['r2'] for s in scores[alg]])
    ]
    for alg in scores
}, index=['MSE', 'MAE', 'R?'])


mean_results.to_csv("/home/flopes/artigo_matheus_jan26/2026_bracis_datasets/results/results_MOU2D/regression_results_mean.csv")
std_results.to_csv("/home/flopes/artigo_matheus_jan26/2026_bracis_datasets/results/results_MOU2D/regression_results_std.csv")

pd.DataFrame(best_params).T.to_csv("/home/flopes/artigo_matheus_jan26/2026_bracis_datasets/results/results_MOU2D/regression_best_params.csv")

print("M?dia das m?tricas (? MSE/MAE melhor, ? R? melhor)")
display(mean_results.round(4))

print("Desvio padr?o das m?tricas")
display(std_results.round(4))

## MLP Regression (with Keras)

In [ ]:
import os

# Aponta para a pasta 'lib' do seu ambiente, onde criamos a pasta 'nvvm'
# O TensorFlow vai automaticamente procurar por 'nvvm/libdevice/libdevice.10.bc' dentro deste caminho
os.environ['XLA_FLAGS'] = "--xla_gpu_cuda_data_dir=/home/flopes/miniconda3/envs/gpu_env/lib"

# Opcional: Garante que as bibliotecas do CUDA (cudart, cublas) sejam achadas
os.environ['LD_LIBRARY_PATH'] = f"/home/flopes/miniconda3/envs/gpu_env/lib:{os.environ.get('LD_LIBRARY_PATH', '')}"

import tensorflow as tf

In [ ]:
import time
import numpy as np
import tensorflow as tf
import keras_tuner as kt
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.regularizers import l2

import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:

def build_mlp(hp, n_features, n_outputs):
    model = Sequential()

    # N?mero de camadas ocultas
    n_layers = hp.Int("n_layers", min_value=1, max_value=3)

    for i in range(n_layers):
        # N?mero de unidades na camada
        units = hp.Int(f"units_{i}", min_value=64, max_value=512, step=64)

        # Fun??o de ativa??o
        activation = hp.Choice(f"activation_{i}", ["relu", "tanh", "swish", "gelu"])

        # Camada densa com L2
        model.add(
            Dense(
                units,
                activation=activation,
                kernel_regularizer=l2(hp.Choice("l2", [0.0, 1e-4, 1e-3])),
                input_shape=(n_features,) if i == 0 else None
            )
        )

        # Batch Normalization opcional
        if hp.Boolean("batch_norm"):
            model.add(BatchNormalization())

        # Dropout
        model.add(Dropout(hp.Float("dropout", min_value=0.0, max_value=0.5, step=0.1)))

    # Camada de sa?da (regress?o multi-output)
    model.add(Dense(n_outputs, activation="linear"))

    # Escolha do otimizador
    optimizer_name = hp.Choice("optimizer", ["adam", "rmsprop"])
    lr = hp.Choice("lr", [1e-2, 5e-3, 1e-3, 5e-4, 1e-4])

    optimizer = Adam(learning_rate=lr) if optimizer_name == "adam" else RMSprop(learning_rate=lr)

    model.compile(optimizer=optimizer, loss="mse")

    return model

In [ ]:
def run_mlp_cv_keras_tuner(
    X,
    y,
    preprocessor,
    n_splits=5,
    max_trials=10,
    epochs=100,
    batch_size=32
):

    scores = []
    best_params = {}
    times = {}

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    start_total = time.time()
    search_time = 0
    cv_time = 0

    for fold, (train_idx, test_idx) in enumerate(cv.split(X), 1):

        print(f"\nMLP ? Fold {fold}/{n_splits}")

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Preprocessamento (fit apenas no treino)
        X_train_p = preprocessor.fit_transform(X_train)
        X_test_p  = preprocessor.transform(X_test)

        tuner = kt.RandomSearch(
            lambda hp: build_mlp(
                hp,
                n_features=X_train_p.shape[1],
                n_outputs=y.shape[1]
            ),
            objective="val_loss",
            max_trials=max_trials,
            directory="keras_tuner",
            project_name=f"mlp_fold_{fold}",
            overwrite=True
        )

        start_search = time.time()
        tuner.search(
            X_train_p,
            y_train,
            validation_split=0.2,
            epochs=epochs,
            batch_size=batch_size,
            verbose=0
        )
        search_time += time.time() - start_search

        best_model = tuner.get_best_models(1)[0]
        best_hp = tuner.get_best_hyperparameters(1)[0].values

        start_cv = time.time()
        y_pred = best_model.predict(X_test_p, verbose=0)
        cv_time += time.time() - start_cv

        scores.append({
            "mse": mean_squared_error(y_test, y_pred, multioutput="uniform_average"),
            "mae": mean_absolute_error(y_test, y_pred, multioutput="uniform_average"),
            "r2":  r2_score(y_test, y_pred, multioutput="uniform_average")
        })

        best_params[f"fold_{fold}"] = best_hp

        tf.keras.backend.clear_session()

    total_time = time.time() - start_total

    times = {
        "search_time_min": search_time / 60,
        "cv_time_min": cv_time / 60,
        "total_time_min": total_time / 60
    }

    return scores, best_params, times

In [ ]:
scores_mlp, best_params_mlp, times_mlp = run_mlp_cv_keras_tuner(
    X=X,
    y=y,
    preprocessor=preprocessor,
    n_splits=5
)

In [ ]:
display(scores_mlp)
display(times_mlp)

In [ ]:
scores_2 = {}
best_params_2 = {}
times_2 = {}

scores_2["MLP"] = scores_mlp
best_params_2["MLP"] = best_params_mlp
times_2["MLP"] = times_mlp

times_df = pd.DataFrame(times_2).T.round(2)
times_df.to_csv("/home/flopes/artigo_matheus_jan26/2026_bracis_datasets/results/results_MOU2D/regression_times_MLP.csv")

print("\nTempo de execu??o por algoritmo (minutos)")
display(times_df)

mean_results_2 = pd.DataFrame({
    alg: [
        np.mean([s['mse'] for s in scores_2[alg]]),
        np.mean([s['mae'] for s in scores_2[alg]]),
        np.mean([s['r2'] for s in scores_2[alg]])
    ]
    for alg in scores_2
}, index=['MSE', 'MAE', 'R?'])

std_results_2 = pd.DataFrame({
    alg: [
        np.std([s['mse'] for s in scores_2[alg]]),
        np.std([s['mae'] for s in scores_2[alg]]),
        np.std([s['r2'] for s in scores_2[alg]])
    ]
    for alg in scores_2
}, index=['MSE', 'MAE', 'R?'])

mean_results_2.to_csv("/home/flopes/artigo_matheus_jan26/2026_bracis_datasets/results/results_MOU2D/regression_results_mean_MLP.csv")
std_results_2.to_csv("/home/flopes/artigo_matheus_jan26/2026_bracis_datasets/results/results_MOU2D/regression_results_std_MLP.csv")
pd.DataFrame(best_params_2).T.to_csv("/home/flopes/artigo_matheus_jan26/2026_bracis_datasets/results/results_MOU2D/regression_best_params_MLP.csv")
pd.DataFrame(best_params_2["MLP"]).T.to_csv("/home/flopes/artigo_matheus_jan26/2026_bracis_datasets/results/results_MOU2D/mlp_best_params_per_fold.csv")

print("M?dia das m?tricas (? MSE/MAE melhor, ? R? melhor)")
display(mean_results_2.round(4))

print("Desvio padr?o das m?tricas")
display(std_results_2.round(4))